# Lab 5 Agent Loop
## MM my own implementation

In [7]:
# imports
from openai import OpenAI
from dotenv import load_dotenv
import json

load_dotenv(override=True)
client = OpenAI()

In [8]:
# helper fns
todos = []  # [{"desc": "...", "done": False}, ...]

def get_todo_report() -> str:
    lines = []
    for i, todo in enumerate(todos, 1):
        mark = "DONE" if todo["done"] else "   "
        lines.append(f"#{i}: [{mark}] {todo['description']}")
    return "\n".join(lines)

def create_todos(descriptions: list[str]) -> str:
    for desc in descriptions:
        todos.append({"description": desc, "done": False})
    return get_todo_report()

def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos):
        todos[index - 1]["done"] = True
        print(completion_notes)
    else:
        return "Invalid index."
    return get_todo_report()

In [9]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "create_todos",
            "description": "Add new todos and return the full list",
            "parameters": {
                "type": "object",
                "properties": {
                    "descriptions": {"type": "array", "items": {"type": "string"}}
                },
                "required": ["descriptions"],
                "additionalProperties": False,
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "mark_complete",
            "description": "Mark the todo at the given 1-based index as done and return the full list",
            "parameters": {
                "type": "object",
                "properties": {
                    "index": {"type": "integer"},
                    "completion_notes": {"type": "string"},
                },
                "required": ["index", "completion_notes"],
                "additionalProperties": False,
            },
        },
    },
]

TOOL_MAP = {
    "create_todos": create_todos,
    "mark_complete": mark_complete,
}

In [10]:
def loop(messages):
    while True:
        response = client.chat.completions.create(
            model="gpt-5.2", messages=messages, tools=tools
        )
        choice = response.choices[0]
        if choice.finish_reason == "tool_calls":
            messages.append(choice.message)
            for tc in choice.message.tool_calls:
                tool_fn = TOOL_MAP.get(tc.function.name)
                if tool_fn is None:
                    result = "Unknown tool"
                else:
                    args = json.loads(tc.function.arguments)
                    result = tool_fn(**args)

                # print the todo report so you see the outline / progress
                if isinstance(result, str):
                    print(result)

                messages.append({
                    "role": "tool",
                    "content": json.dumps(result),
                    "tool_call_id": tc.id,
                })
        else:
            print(choice.message.content)
            break

In [11]:
system_message = """
You solve problems by first creating a todo plan, then carrying out each step.
After each step, mark it complete. Reply with the final answer only.
"""
user_message = """
A train leaves Boston at 2pm at 60mph. Another leaves New York at 3pm at 80mph toward Boston.
When do they meet? (Assume Boston-NY distance is 215 miles.)
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message},
]
loop(messages)

#1: [   ] Compute head start distance of Boston train by 3pm
#2: [   ] Set up relative speed after 3pm and solve time to meet
#3: [   ] Convert meeting time to clock time and present result
From 2pm to 3pm is 1 hour at 60 mph, so Boston train travels 60 miles toward NY by 3pm.
#1: [DONE] Compute head start distance of Boston train by 3pm
#2: [   ] Set up relative speed after 3pm and solve time to meet
#3: [   ] Convert meeting time to clock time and present result
Remaining separation at 3pm: 215-60=155 miles. Closing speed after 3pm: 60+80=140 mph. Time to meet after 3pm: 155/140=1.107142... hours = 1 h 6 min 26 s.
#1: [DONE] Compute head start distance of Boston train by 3pm
#2: [DONE] Set up relative speed after 3pm and solve time to meet
#3: [   ] Convert meeting time to clock time and present result
Meeting time = 3:00pm + 1:06:26 ≈ 4:06pm (about 4:06:26pm).
#1: [DONE] Compute head start distance of Boston train by 3pm
#2: [DONE] Set up relative speed after 3pm and solve time to m